# CSF-CUF · Interactive Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/quickstart_colab_interactive.ipynb)

**Run a CUF analysis without writing Python or editing YAML files.** This notebook uses the existing [CSF-CUF Quick Start](https://github.com/giovanniboscu/continuous-section-field/tree/main/cuf/quickstart): **model** (geometry and materials), **problem** (load and boundary conditions), and **case** (CUF and longitudinal approximations).

**First time:** choose **Runtime → Run all**. Later, change any parameter in step 2 and run that cell again. The notebook automatically updates temporary YAML files, runs the solver and displays the response. The original GitHub files are not modified.

### 1 · Prepare the environment
Run once per new Colab runtime. This clones the public repository and installs the solver and YAML support.

In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil

REPO = Path('/content/continuous-section-field')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/giovanniboscu/continuous-section-field.git', str(REPO)], check=True)
else:
    # Pick up files published after this Colab runtime first cloned the repo.
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'pypardiso', 'pyyaml'], check=True)
QUICKSTART = REPO / 'cuf' / 'quickstart'
assert (QUICKSTART / 'models/rectangular_prismatic_csf.yaml').is_file()
assert (QUICKSTART / 'problems/torsion_halfwave.yaml').is_file()
assert (QUICKSTART / 'cases/torsion_halfwave_legendre_N02.yaml').is_file()
assert (QUICKSTART / 'models/actions_colab.yaml').is_file(), 'Publish actions_colab.yaml to cuf/quickstart/models/ on GitHub.'
print('CSF-CUF ready. Choose parameters below and press Run.')

### 2 · Configure, solve and visualize

Change the parameters and press **▶**. The notebook updates temporary YAML files, runs CSF-CUF, and uses **csf-actions** to display S0/S1, section properties and material variation.


In [ ]:
# @title Configure model and run CSF-CUF { display-mode: "form" }
# @markdown **Numerical approximation**
cuf_order = 2 # @param {type:"integer"}
longitudinal_order = 6 # @param {type:"integer"}
# @markdown **Material (Young's modulus, MPa)**
E_S0 = 71700.0 # @param {type:"number"}
E_S1 = 71700.0 # @param {type:"number"}
# @markdown **Rectangular section dimensions (mm)**
width_S0 = 100.0 # @param {type:"number"}
height_S0 = 100.0 # @param {type:"number"}
width_S1 = 100.0 # @param {type:"number"}
height_S1 = 100.0 # @param {type:"number"}

from pathlib import Path
import subprocess
import shutil
import os
import yaml
from IPython.display import display, Markdown

if not (isinstance(cuf_order, int) and 1 <= cuf_order <= 20):
    raise ValueError('CUF order must be an integer from 1 to 20.')
if not (isinstance(longitudinal_order, int) and 1 <= longitudinal_order <= 20):
    raise ValueError('Longitudinal order must be an integer from 1 to 20.')
for label, value in [('E_S0', E_S0), ('E_S1', E_S1),
                     ('width_S0', width_S0), ('height_S0', height_S0),
                     ('width_S1', width_S1), ('height_S1', height_S1)]:
    if not (0 < float(value) < float('inf')):
        raise ValueError(f'{label} must be positive and finite.')

# Work exclusively on disposable copies; preserve original relative YAML paths.
WORK = Path('/content/csf_cuf_quickstart_work')
for folder in ('models', 'problems', 'cases'):
    (WORK / folder).mkdir(parents=True, exist_ok=True)
for folder, filename in (
    ('models', 'rectangular_prismatic_csf.yaml'),
    ('problems', 'torsion_halfwave.yaml'),
    ('cases', 'torsion_halfwave_legendre_N02.yaml'),
):
    shutil.copy2(QUICKSTART / folder / filename, WORK / folder / filename)

model_path = WORK / 'models/rectangular_prismatic_csf.yaml'
case_path = WORK / 'cases/torsion_halfwave_legendre_N02.yaml'
model = yaml.safe_load(model_path.read_text(encoding='utf-8'))
case = yaml.safe_load(case_path.read_text(encoding='utf-8'))

def rectangle(width, height):
    w, h = width / 2.0, height / 2.0
    return [[-w, -h], [w, -h], [w, h], [-w, h]]

for station, E, width, height in (
    ('S0', E_S0, width_S0, height_S0),
    ('S1', E_S1, width_S1, height_S1),
):
    polygons = model['CSF']['sections'][station]['polygons']
    assert len(polygons) == 1, f'Expected one rectangle in {station}'
    polygons[0]['weight'] = float(E)
    polygons[0]['vertices'] = rectangle(float(width), float(height))

case['cuf']['order'] = cuf_order
case['longitudinal']['order'] = longitudinal_order
model_path.write_text(yaml.safe_dump(model, sort_keys=False), encoding='utf-8')
case_path.write_text(yaml.safe_dump(case, sort_keys=False), encoding='utf-8')

print(f'Running CUF N{cuf_order:02d}; longitudinal order {longitudinal_order}')
print(f'E: S0={E_S0:g}, S1={E_S1:g} MPa')
print(f'Geometry: S0={width_S0:g} x {height_S0:g}, S1={width_S1:g} x {height_S1:g} mm')
print('Solving...')
run = subprocess.run(
    ['csf-cuf', 'cases/torsion_halfwave_legendre_N02.yaml'],
    cwd=WORK, capture_output=True, text=True,
)
if run.stdout.strip():
    print(run.stdout)
if run.returncode != 0:
    if run.stderr.strip():
        print(run.stderr)
    raise RuntimeError(f'csf-cuf failed (exit code {run.returncode})')
if run.stderr.strip():
    print(run.stderr)

RESULTS = WORK / 'output/torsion_halfwave_legendre_N02'
response_path = RESULTS / 'response.txt'
if not response_path.is_file():
    raise FileNotFoundError(f'Solver completed but response.txt was not found: {response_path}')
response_text = response_path.read_text(encoding='utf-8')
display(Markdown('### Structural response (`response.txt`)'))
print(response_text)

# Keep previous response in this runtime for a simple side-by-side comparison.
previous = globals().get('_previous_csf_response')
if previous is not None:
    display(Markdown('### Previous run (for comparison)'))
    print(previous)
_previous_csf_response = response_text
print('Results:', RESULTS)
print('Original GitHub YAML files remain unchanged.')
# CSF model visualization and properties (official csf-actions).
# actions_colab.yaml is kept alongside the original model on GitHub.
action_source = QUICKSTART / 'models' / 'actions_colab.yaml'
if not action_source.is_file():
    # The Colab clone may predate the action file; refresh it automatically.
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    if not action_source.is_file():
        raise FileNotFoundError(
            'actions_colab.yaml is not present in the public GitHub repository: '
            'cuf/quickstart/models/actions_colab.yaml'
        )
shutil.copy2(action_source, WORK / 'models' / 'actions_colab.yaml')
plot_dir = WORK / 'models' / 'out'
plot_dir.mkdir(parents=True, exist_ok=True)
for old_plot in plot_dir.glob('*.png'):
    old_plot.unlink()
for old_csv in plot_dir.glob('*.csv'):
    old_csv.unlink()
plot_env = os.environ.copy()
plot_env['MPLBACKEND'] = 'Agg'
plot_run = subprocess.run(
    ['csf-actions', 'rectangular_prismatic_csf.yaml', 'actions_colab.yaml'],
    cwd=WORK / 'models', env=plot_env,
    capture_output=True, text=True,
)
print('csf-actions exit code:', plot_run.returncode)
if plot_run.stdout.strip():
    print(plot_run.stdout)
if plot_run.stderr.strip():
    print(plot_run.stderr)
if plot_run.returncode != 0:
    raise RuntimeError('CSF actions failed')
from IPython.display import Image
for title, filename in (
    ('S0 and S1 sections', 'sections_S0_S1.png'),
    ('Area and moments of inertia along the beam', 'section_properties.png'),
    ('Material variation', 'material_variation.png'),
):
    image_path = plot_dir / filename
    display(Markdown('### ' + title))
    if image_path.is_file():
        display(Image(filename=str(image_path)))
    else:
        print('Image not created:', image_path)
csv_path = plot_dir / 'section_properties.csv'
display(Markdown('### Section properties at S0 and S1'))
if csv_path.is_file():
    print(csv_path.read_text(encoding='utf-8'))
else:
    print('CSV not created:', csv_path)


**Done.** Change any parameter in step 2 and run it again. The figures and numerical properties are regenerated from the modified CSF model. Source YAML files on GitHub are unchanged.
